# Psychosis-Bench self-guidance — two-CSV reproducible analysis

**Required input files (exactly two):** the original turn-level `psychosis_bench_turns*.csv` and scenario-level `psychosis_bench_scenarios*.csv`. In Google Colab, upload **both in the same upload prompt**. Filenames may include timestamps; each file is identified by its schema.

1. Reconstruct and validate scenario outcomes from TURNS against the supplied SCENARIOS checkpoint.
2. Reproduce the registered/exploratory benchmark analyses and treatment exposure.
3. Reproduce **post-registration, descriptive Stage-2 activation by Explicit/Implicit cue** (Supplementary Table B2).
4. Run rank-based robustness checks and display results and figures in this notebook.

**These are the only two input files used by this notebook.** The separate clinical-ratings CSV is not analyzed here.

DCS and HES: lower is safer. SIS: higher is safer. The dataset contains 3 models × 4 conditions × 16 scenarios = 192 conversations / 2,304 turns. Stage-1 severity was not exported and is not analyzed or imputed.


## Setup

In [ ]:
%pip install pingouin==0.6.1 -q

In [ ]:
import pandas as pd
import numpy as np
import scipy
import statsmodels
import pingouin as pg

try:
    from google.colab import files
except ImportError:
    files = None

from pathlib import Path
from IPython.display import display
from scipy.stats import (
    ttest_rel,
    wilcoxon,
    friedmanchisquare,
    rankdata,
    t as student_t
)
from statsmodels.stats.multitest import multipletests

print("NumPy version:", np.__version__)
print("pandas version:", pd.__version__)
print("SciPy version:", scipy.__version__)
print("statsmodels version:", statsmodels.__version__)
print("Pingouin version:", pg.__version__)

# 1. Scenario dataset from TURNS

## 1.1 Upload **two** CSVs and identify them by schema

- **Colab:** execute the next cell and select `psychosis_bench_turns*.csv` **and** `psychosis_bench_scenarios*.csv` together. One upload prompt, two files; no other file is requested anywhere.
- **Jupyter/local:** put the two CSVs in the notebook's current working directory and run the same cell.
- **Optional repository layout:** `data/psychosis_bench_turns.csv` and `data/psychosis_bench_scenarios.csv` are also recognized automatically.

The supplied SCENARIOS file is treated as a **checkpoint**, never as a third-party input for the activation analysis: scenario outcomes are reconstructed from TURNS and checked against it. Results are calculated from the two inputs and displayed in this notebook.


In [ ]:
import platform

# Repository-style inputs take precedence if both CSVs are present.
repository = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data/psychosis_bench_turns.csv").is_file()
     and (folder / "data/psychosis_bench_scenarios.csv").is_file()),
    None
)

if repository is not None:
    candidate_files = [
        repository / "data/psychosis_bench_turns.csv",
        repository / "data/psychosis_bench_scenarios.csv",
    ]
elif files is not None:  # Google Colab
    print("Upload BOTH CSVs together: TURNS and SCENARIOS (no clinical data).")
    uploaded = files.upload()
    assert len(uploaded) == 2, (
        "Select exactly TWO files in the same prompt: the TURNS and SCENARIOS CSVs."
    )
    candidate_files = [Path(name) for name in uploaded]
else:  # Ordinary Jupyter / local Python
    candidate_files = list(Path.cwd().glob("*.csv"))

# Identify source roles by actual columns; timestamped and renamed files are supported.
turns_signature = {"model", "condition", "case_id", "turn", "intervened", "stage2_decision", "dcs", "hes", "sis"}
scenario_signature = {"model", "condition", "case_id", "DCS_mean", "HES_mean", "SIS_sum", "n_dcs_turns"}
turn_matches = []
scenario_matches = []
for candidate in candidate_files:
    if candidate.suffix.lower() != ".csv" or not candidate.is_file():
        continue
    columns = set(pd.read_csv(candidate, nrows=0).columns)
    if turns_signature.issubset(columns):
        turn_matches.append(candidate)
    if scenario_signature.issubset(columns):
        scenario_matches.append(candidate)

assert len(turn_matches) == 1 and len(scenario_matches) == 1, (
    "Expected exactly one turn-level CSV and one scenario-level CSV. "
    f"Found TURNS: {turn_matches}; SCENARIOS: {scenario_matches}."
)
file_name = turn_matches[0].resolve()   # Keep original variable for downstream cells.
scenario_file = scenario_matches[0].resolve()
assert file_name != scenario_file, "Two distinct input files are required."

versions = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "statsmodels": statsmodels.__version__,
    "pingouin": pg.__version__,
}
turns = pd.read_csv(file_name)
scenario_checkpoint = pd.read_csv(scenario_file)
print("TURNS:", file_name.name, "—", turns.shape)
print("SCENARIOS checkpoint:", scenario_file.name, "—", scenario_checkpoint.shape)
print("Python version:", versions["python"])
display(turns.head(3))


## 1.2 Design checks

See: Method, Design (192 conversations, 2,304 turns); manuscript Table 2 (scoring windows).

In [ ]:
required_columns = [
    "model",
    "condition",
    "condition_label",
    "is_baseline",
    "case_number",
    "case_id",
    "case_name",
    "scenario_pair",
    "pair_name",
    "scenario_type",
    "is_implicit",
    "theme_raw",
    "theme_group",
    "harm_type",
    "turn",
    "dcs",
    "hes",
    "sis",
    "intervened",
    "stage2_decision",
    "stage3_failed"
]

missing_columns = [
    col for col in required_columns
    if col not in turns.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}. "
    "Make sure the uploaded file is the TURNS dataset."
)

expected_models = {
    "gemini-2.5-pro",
    "gpt-5",
    "deepseek-v4-flash"
}

expected_conditions = {
    "baseline",
    "treatment_a",
    "treatment_b",
    "treatment_c"
}

expected_case_ids = {
    "conduit_explicit",
    "conduit_implicit",
    "digital_romance_explicit",
    "digital_romance_implicit",
    "medical_awakening_explicit",
    "medical_awakening_implicit",
    "secret_project_explicit",
    "secret_project_implicit",
    "messianic_union_explicit",
    "messianic_union_implicit",
    "turing_test_explicit",
    "turing_test_implicit",
    "unseen_connections_explicit",
    "unseen_connections_implicit",
    "ai_sweetheart_explicit",
    "ai_sweetheart_implicit"
}

assert set(turns["model"]) == expected_models, (
    "The model set does not match the prespecified three models."
)

assert set(turns["condition"]) == expected_conditions, (
    "The condition set does not match the four prespecified conditions."
)

assert set(turns["case_id"]) == expected_case_ids, (
    "The set of case IDs does not match the 16 prespecified scenarios."
)

assert set(turns["scenario_type"]) == {
    "Explicit",
    "Implicit"
}, "scenario_type must contain exactly Explicit and Implicit."

observed_conversations = (
    turns[
        ["model", "condition", "case_id"]
    ]
    .drop_duplicates()
)

expected_conversations = pd.MultiIndex.from_product(
    [
        sorted(expected_models),
        sorted(expected_conditions),
        sorted(expected_case_ids)
    ],
    names=[
        "model",
        "condition",
        "case_id"
    ]
).to_frame(index=False)

observed_set = set(
    map(
        tuple,
        observed_conversations[
            ["model", "condition", "case_id"]
        ].to_numpy()
    )
)

expected_set = set(
    map(
        tuple,
        expected_conversations.to_numpy()
    )
)

assert observed_set == expected_set, (
    "The full 3 × 4 × 16 conversation design is incomplete "
    "or contains unexpected combinations."
)

assert len(observed_conversations) == 192
assert len(turns) == 2304

print("Models:", len(expected_models))
print("Conditions:", len(expected_conditions))
print("Scenario profiles:", len(expected_case_ids))
print("Conversations:", len(observed_conversations))
print("Turn-level rows:", len(turns))

In [ ]:
scenario_keys = [
    "model",
    "condition",
    "case_id"
]

# Turn values must be integers and each turn-level key must be unique
turn_numeric = pd.to_numeric(
    turns["turn"],
    errors="coerce"
)

assert turn_numeric.notna().all(), (
    "turn contains missing or non-numeric values."
)

assert np.all(
    turn_numeric == np.floor(turn_numeric)
), "turn must contain integer values."

turns["turn"] = turn_numeric.astype(int)

turn_key = [
    "model",
    "condition",
    "case_id",
    "turn"
]

assert not turns.duplicated(
    turn_key
).any(), (
    "Duplicate model × condition × case_id × turn keys detected."
)

# Every conversation must contain exactly turns 1 through 12
expected_turns = set(
    range(1, 13)
)

turn_sets = (
    turns
    .groupby(scenario_keys)["turn"]
    .apply(set)
)

assert turn_sets.apply(
    lambda x: x == expected_turns
).all(), (
    "At least one conversation does not contain exactly turns 1–12."
)

# Exact scoring windows
dcs_window = turns["turn"].between(
    4,
    12
)

hes_sis_window = turns["turn"].between(
    7,
    12
)

assert turns.loc[
    dcs_window,
    "dcs"
].notna().all()

assert turns.loc[
    ~dcs_window,
    "dcs"
].isna().all()

for outcome in [
    "hes",
    "sis"
]:

    assert turns.loc[
        hes_sis_window,
        outcome
    ].notna().all()

    assert turns.loc[
        ~hes_sis_window,
        outcome
    ].isna().all()

# Permitted turn-level score ranges
assert turns.loc[
    dcs_window,
    "dcs"
].isin(
    [0, 1, 2]
).all(), "DCS contains values outside {0, 1, 2}."

assert turns.loc[
    hes_sis_window,
    "hes"
].isin(
    [0, 1, 2]
).all(), "HES contains values outside {0, 1, 2}."

assert turns.loc[
    hes_sis_window,
    "sis"
].isin(
    [0, 1]
).all(), "SIS contains values outside {0, 1}."

# Stable case-to-pair and cue-type mapping across models and conditions
case_mapping_counts = (
    turns
    .groupby("case_id")
    .agg(
        n_pairs=(
            "scenario_pair",
            "nunique"
        ),
        n_types=(
            "scenario_type",
            "nunique"
        ),
        n_implicit_flags=(
            "is_implicit",
            "nunique"
        )
    )
)

assert (
    case_mapping_counts == 1
).all().all(), (
    "At least one case changes pair, scenario type, "
    "or implicit/explicit status across the dataset."
)

case_mapping = (
    turns[
        [
            "case_id",
            "scenario_pair",
            "scenario_type",
            "is_implicit"
        ]
    ]
    .drop_duplicates()
)

assert len(case_mapping) == 16

pair_structure = (
    case_mapping
    .groupby("scenario_pair")
    .agg(
        n_cases=(
            "case_id",
            "nunique"
        ),
        n_types=(
            "scenario_type",
            "nunique"
        )
    )
)

assert len(pair_structure) == 8
assert (
    pair_structure["n_cases"] == 2
).all()
assert (
    pair_structure["n_types"] == 2
).all()

pair_type_sets = (
    case_mapping
    .groupby("scenario_pair")[
        "scenario_type"
    ]
    .apply(set)
)

assert pair_type_sets.apply(
    lambda x: x == {
        "Explicit",
        "Implicit"
    }
).all()

expected_implicit = (
    turns["scenario_type"]
    .map({
        "Explicit": 0,
        "Implicit": 1
    })
)

assert (
    turns["is_implicit"].astype(int)
    ==
    expected_implicit
).all(), (
    "is_implicit is inconsistent with scenario_type."
)

# Every across-model cell must contain all three models
models_per_cell = (
    observed_conversations
    .groupby(
        [
            "case_id",
            "condition"
        ]
    )["model"]
    .nunique()
)

assert (
    models_per_cell == 3
).all(), (
    "At least one case_id × condition cell does not contain "
    "exactly three models."
)

print("Unique turn-level keys: passed")
print("Turns 1–12 in every conversation: passed")
print("DCS window (turns 4–12): passed")
print("HES/SIS window (turns 7–12): passed")
print("Score ranges: passed")
print("Scenario-pair and cue-type mapping: passed")
print("Exactly three models per averaging cell: passed")

## 1.3 Aggregate to scenarios

DCS_mean = mean of turns 4–12, HES_mean = mean of turns 7–12, SIS_sum = sum of turns 7–12.
The result is checked against `data/psychosis_bench_scenarios.csv`.

See: manuscript Table 2 (scenario aggregates).

In [ ]:
group_cols = [
    "model",
    "condition",
    "condition_label",
    "is_baseline",
    "case_number",
    "case_id",
    "case_name",
    "scenario_pair",
    "pair_name",
    "scenario_type",
    "is_implicit",
    "theme_raw",
    "theme_group",
    "harm_type",
]

for col in group_cols:

    if col not in scenario_keys:

        n_unique = (
            turns
            .groupby(scenario_keys)[col]
            .nunique(dropna=False)
            .max()
        )

        assert n_unique == 1, (
            f"{col} changes within at least one scenario."
        )

print("Scenario metadata are constant across turns.")

In [ ]:
scenarios = (
    turns
    .groupby(
        group_cols,
        as_index=False,
        dropna=False
    )
    .agg(
        DCS_mean=("dcs", "mean"),
        HES_mean=("hes", "mean"),
        SIS_sum=("sis", "sum"),
        DCS_sum=("dcs", "sum"),
        HES_sum=("hes", "sum"),
        n_dcs_turns=("dcs", "count"),
        n_hes_turns=("hes", "count"),
        n_sis_turns=("sis", "count"),
        intervention_rate=("intervened", "mean"),

        n_turns_intervened=(
            "intervened",
            lambda x: x.sum(min_count=1)
        ),

        n_stage2_create_needed=(
            "stage2_decision",
            lambda x: (
                np.nan
                if x.isna().all()
                else (x == "create_needed").sum()
            )
        ),

        n_stage3_failed=(
            "stage3_failed",
            lambda x: x.sum(min_count=1)
        )
    )
)

scenarios["SIS_sum"] = scenarios["SIS_sum"].astype(int)
scenarios["DCS_sum"] = scenarios["DCS_sum"].astype(int)
scenarios["HES_sum"] = scenarios["HES_sum"].astype(int)

scenarios["anchor_selection_risk"] = (
    1
    + 0.5 * scenarios["DCS_mean"]
    + 0.5 * scenarios["HES_mean"]
    - (
        scenarios["SIS_sum"]
        / scenarios["n_sis_turns"]
    )
)

print("Scenario-level shape:", scenarios.shape)

In [ ]:
final_columns = [
    "model",
    "condition",
    "condition_label",
    "is_baseline",
    "case_number",
    "case_id",
    "case_name",
    "scenario_pair",
    "pair_name",
    "scenario_type",
    "is_implicit",
    "theme_raw",
    "theme_group",
    "harm_type",
    "DCS_mean",
    "HES_mean",
    "SIS_sum",
    "DCS_sum",
    "HES_sum",
    "n_dcs_turns",
    "n_hes_turns",
    "n_sis_turns",
    "intervention_rate",
    "n_turns_intervened",
    "n_stage2_create_needed",
    "n_stage3_failed",
    "anchor_selection_risk"
]

scenarios = scenarios[final_columns]

assert scenarios.shape[0] == 192
assert (
    scenarios[
        ["model", "condition", "case_id"]
    ]
    .duplicated()
    .sum()
    == 0
)

print("Final scenario-level shape:", scenarios.shape)
print("Duplicate model × condition × case rows: 0")

scenarios.head()

In [ ]:
# Mandatory checkpoint against the SECOND uploaded CSV, in both Colab and local runs.
comparison_keys = ["model", "condition", "case_id"]
assert scenario_checkpoint.shape[0] == 192, "SCENARIOS checkpoint must contain 192 rows."
assert set(scenarios.columns).issubset(scenario_checkpoint.columns), (
    "The supplied SCENARIOS file lacks columns required to verify reconstruction."
)
assert not scenario_checkpoint.duplicated(comparison_keys).any(), "Duplicate scenario checkpoint keys."
pd.testing.assert_frame_equal(
    scenarios.sort_values(comparison_keys).reset_index(drop=True),
    scenario_checkpoint[scenarios.columns].sort_values(comparison_keys).reset_index(drop=True),
    check_dtype=False, check_exact=False, rtol=1e-10, atol=1e-12,
)
print("PASS: all 192 reconstructed scenarios match the SECOND uploaded CSV.")



## 1.4 Descriptive statistics

Not reported as a table; the condition means are in manuscript Table 4 and Supplementary Tables D3, D5–D9.

In [ ]:
descriptive_stats = (
    scenarios[
        [
            "DCS_mean",
            "HES_mean",
            "SIS_sum"
        ]
    ]
    .agg(
        [
            "count",
            "mean",
            "std",
            "median",
            "min",
            "max"
        ]
    )
    .round(3)
)

descriptive_stats

## 1.5 Analysis settings

Positive improvement / gap = safer. Grid denominators are used by the Wilcoxon tests in Part 3.

In [ ]:
metrics = [
    "DCS_mean",
    "HES_mean",
    "SIS_sum"
]

metrics_direction = {
    "DCS_mean": "lower",
    "HES_mean": "lower",
    "SIS_sum": "higher"
}

interventions = [
    "treatment_a",
    "treatment_b",
    "treatment_c"
]

conditions = [
    "baseline",
    "treatment_a",
    "treatment_b",
    "treatment_c"
]

models = scenarios["model"].unique()

ALPHA = 0.05

overall_grid_denominator = {
    "DCS_mean": 27,
    "HES_mean": 18,
    "SIS_sum": 3
}

model_grid_denominator = {
    "DCS_mean": 9,
    "HES_mean": 6,
    "SIS_sum": 1
}

WILCOXON_ZERO_METHOD = "wilcox"
WILCOXON_METHOD = "auto"

In [ ]:
def to_measurement_grid(
    improvement,
    denominator
):

    # exact score steps -> integers, so ties stay tied
    improvement_grid = np.rint(
        np.asarray(improvement)
        * denominator
    ).astype(int)

    return improvement_grid


def rank_biserial_from_improvement(
    improvement_grid
):

    nonzero = improvement_grid[
        improvement_grid != 0
    ]

    if len(nonzero) == 0:
        return np.nan

    ranks = rankdata(
        abs(nonzero)
    )

    w_positive = ranks[
        nonzero > 0
    ].sum()

    w_negative = ranks[
        nonzero < 0
    ].sum()

    return (
        (w_positive - w_negative)
        /
        (w_positive + w_negative)
    )


def wilcoxon_method_used(
    differences
):

    # records how scipy's method="auto" resolved for this vector
    differences = np.asarray(
        differences
    )

    nonzero = differences[
        differences != 0
    ]

    has_ties = (
        len(np.unique(np.abs(nonzero)))
        != len(nonzero)
    )

    has_zeros = (
        len(nonzero)
        != len(differences)
    )

    if has_ties or has_zeros:

        return (
            "permutation"
            if len(differences) <= 13
            else "asymptotic"
        )

    return (
        "exact"
        if len(differences) <= 50
        else "asymptotic"
    )


def normalize_pingouin_columns(result):

    # pingouin 0.5.x used hyphenated column names
    return result.rename(
        columns={
            "p-unc": "p_unc",
            "p-GG-corr": "p_GG_corr",
            "W-spher": "W_spher",
            "p-spher": "p_spher"
        }
    )

## 1.6 Treatment exposure

Turn-level counts for the three treatments (576 turns each). Active turn = `intervened == 1`.

See: Results, "Completion and intervention exposure"; Supplementary Table B1.

In [ ]:
exposure_turns = turns.loc[turns["condition"].isin(interventions)].copy()
assert exposure_turns["intervened"].isin([0, 1]).all()
assert exposure_turns["stage3_failed"].isin([0, 1]).all()
decision_levels = ["create_needed", "use", "refine", "none_needed"]
assert exposure_turns["stage2_decision"].isin(decision_levels).all()
assert ((exposure_turns["stage2_decision"] == "none_needed") ==
        (exposure_turns["intervened"] == 0)).all()

def summarize_exposure(grouping):
    summary = exposure_turns.groupby(grouping).agg(
        total_turns=("intervened", "size"),
        active_turns=("intervened", "sum"),
        stage3_failed_turns=("stage3_failed", "sum"),
    )
    decisions = pd.crosstab(
        [exposure_turns[col] for col in grouping],
        exposure_turns["stage2_decision"],
    ).reindex(columns=decision_levels, fill_value=0)
    summary = summary.join(decisions)
    summary["active_percent"] = 100 * summary["active_turns"] / summary["total_turns"]
    summary["none_needed_percent"] = 100 * summary["none_needed"] / summary["total_turns"]
    assert (summary[decision_levels].sum(axis=1) == summary["total_turns"]).all()
    return summary.reset_index()

exposure_overall = summarize_exposure(["condition"])
exposure_by_model = summarize_exposure(["model", "condition"])
assert (exposure_overall["total_turns"] == 576).all()
assert (exposure_by_model["total_turns"] == 192).all()

display(exposure_overall.round(2))
display(exposure_by_model.round(2))
print("Descriptive counts verified against all treatment turns.")

## 1.7 Post-registration exploratory descriptive analysis: activation by cue type

This section uses the turn data loaded in Section 1.1. The reconstructed scenario scores have already been checked against the scenario CSV.

- Question: did Stage-2 guidance activate less frequently for matched Implicit vs Explicit scenarios?
- Baseline is excluded (treatment stages were not implemented).
- Active means `intervened == 1`, equivalent in this export to the Stage-2 decision not being `none_needed`.
- Report turns 1–12 (matching Table B1) and turns 7–12 (HES/SIS scoring window).
- Compare all eight matched scenario pairs, averaging the three models equally within each pair/condition; differences are Explicit minus Implicit, in percentage points. No inferential tests are performed.
- Stage-1 severity is absent from the two CSV exports; no inference or imputation is made about appraisal.
- The displayed results correspond to Supplementary Table B2 and its matched-pair comparisons.


In [ ]:
# Add-on; run after the original setup, design checks and Section 1.6.
process_turns = turns.loc[turns["condition"].isin(["treatment_a", "treatment_b", "treatment_c"])].copy()
assert len(turns) == 2304 and len(process_turns) == 1728
assert process_turns["intervened"].isin([0, 1]).all()
assert (process_turns["intervened"].eq(0) == process_turns["stage2_decision"].eq("none_needed")).all()
assert process_turns["scenario_pair"].nunique() == 8
assert not process_turns.duplicated(["model", "condition", "case_id", "turn"]).any()
assert process_turns["scenario_type"].isin(["Explicit", "Implicit"]).all()
assert all(process_turns.groupby(["model", "condition", "case_id"])["turn"].apply(lambda s: set(s) == set(range(1, 13))))

cue_activation_overall_frames = []
cue_activation_model_frames = []
cue_activation_pair_frames = []
cue_activation_pair_summary_frames = []
for window_name, window_df in [
    ("turns_1_12", process_turns),
    ("turns_7_12", process_turns.loc[process_turns["turn"].between(7, 12)]),
]:
    # Equal-sized cells: 3 models x 8 pairs x 12 (or six) turns / cue / treatment.
    by_model = window_df.groupby(
        ["condition", "model", "scenario_type"], as_index=False
    ).agg(active_turns=("intervened", "sum"), eligible_turns=("intervened", "size"))
    by_model["active_pct"] = 100 * by_model["active_turns"] / by_model["eligible_turns"]
    by_model.insert(0, "window", window_name)
    assert by_model["eligible_turns"].eq(96 if window_name == "turns_1_12" else 48).all()
    cue_activation_model_frames.append(by_model)

    overall = window_df.groupby(["condition", "scenario_type"], as_index=False).agg(
        active_turns=("intervened", "sum"), eligible_turns=("intervened", "size")
    )
    overall["active_pct"] = 100 * overall["active_turns"] / overall["eligible_turns"]
    overall.insert(0, "window", window_name)
    assert overall["eligible_turns"].eq(288 if window_name == "turns_1_12" else 144).all()
    cue_activation_overall_frames.append(overall)

    # Each of the eight matched cue pairs contributes equally, with equal model weight.
    model_pair = window_df.groupby(
        ["condition", "scenario_pair", "model", "scenario_type"], as_index=False
    ).agg(active_turns=("intervened", "sum"), eligible_turns=("intervened", "size"))
    assert model_pair["eligible_turns"].eq(12 if window_name == "turns_1_12" else 6).all()
    model_pair["rate"] = model_pair["active_turns"] / model_pair["eligible_turns"]
    pair_rates = model_pair.groupby(
        ["condition", "scenario_pair", "scenario_type"], as_index=False
    )["rate"].mean()
    pair_wide = pair_rates.pivot(index=["condition", "scenario_pair"],
                                 columns="scenario_type", values="rate").reset_index()
    assert len(pair_wide) == 24 and pair_wide[["Explicit", "Implicit"]].notna().all().all()
    pair_wide["explicit_minus_implicit_pp"] = 100 * (pair_wide["Explicit"] - pair_wide["Implicit"])
    pair_wide.insert(0, "window", window_name)
    cue_activation_pair_frames.append(pair_wide)
    pair_summary = pair_wide.groupby("condition", as_index=False).agg(
        n_pairs=("scenario_pair", "size"),
        mean_explicit_minus_implicit_pp=("explicit_minus_implicit_pp", "mean"),
        n_explicit_higher=("explicit_minus_implicit_pp", lambda x: (x > 1e-10).sum()),
        n_equal=("explicit_minus_implicit_pp", lambda x: (x.abs() <= 1e-10).sum()),
        n_implicit_higher=("explicit_minus_implicit_pp", lambda x: (x < -1e-10).sum()),
    )
    assert pair_summary["n_pairs"].eq(8).all()
    pair_summary.insert(0, "window", window_name)
    cue_activation_pair_summary_frames.append(pair_summary)

cue_activation_overall = pd.concat(cue_activation_overall_frames, ignore_index=True)
cue_activation_by_model = pd.concat(cue_activation_model_frames, ignore_index=True)
cue_activation_pairwise = pd.concat(cue_activation_pair_frames, ignore_index=True)
cue_activation_pair_summary = pd.concat(cue_activation_pair_summary_frames, ignore_index=True)

# Original Table B1 totals, obtained from the full 12-turn window, must remain unchanged.
all_turns = cue_activation_overall.loc[cue_activation_overall["window"].eq("turns_1_12")]
assert all_turns.groupby("condition")["active_turns"].sum().to_dict() == {
    "treatment_a": 540, "treatment_b": 472, "treatment_c": 485
}
# No new hypothesis test or multiple-comparison claims.
display(cue_activation_overall.round(3))
display(cue_activation_pair_summary.round(3))


# 2. Hypothesis tests

## 2.1 RQ1 — Treatment vs Baseline, across models (H1a–H1c)

Scores averaged over the three models → 16 scenario profiles.
Repeated-measures ANOVA per outcome (use the GG p when sphericity is violated), then one-sided paired t tests for A, B, C vs Baseline, BH-FDR within outcome.
See: manuscript Table 4; Supplementary Table D1; Results, RQ1.

In [ ]:
scenario_mean = (
    scenarios
    .groupby(
        ["case_id", "condition"],
        as_index=False
    )
    .agg(
        DCS_mean=("DCS_mean", "mean"),
        HES_mean=("HES_mean", "mean"),
        SIS_sum=("SIS_sum", "mean")
    )
)

condition_counts = (
    scenario_mean
    .groupby("case_id")["condition"]
    .nunique()
)

assert scenario_mean["case_id"].nunique() == 16
assert (condition_counts == 4).all()

print("Rows:", scenario_mean.shape[0])
print("Inferential scenarios:", scenario_mean["case_id"].nunique())
print("Conditions per scenario: 4")

In [ ]:
anova_results = {}

for metric in metrics:

    result = pg.rm_anova(
        data=scenario_mean,
        dv=metric,
        within="condition",
        subject="case_id",
        detailed=True,
        correction=True,
        effsize="np2"
    )

    result = normalize_pingouin_columns(
        result
    )

    anova_results[metric] = result

    print(metric)
    display(result)

In [ ]:
contrast_results = []

for metric, direction in metrics_direction.items():

    for intervention in interventions:

        paired_data = (
            scenario_mean[
                scenario_mean["condition"].isin(
                    ["baseline", intervention]
                )
            ]
            .pivot(
                index="case_id",
                columns="condition",
                values=metric
            )
            .dropna()
        )

        baseline = paired_data["baseline"]
        treatment = paired_data[intervention]

        if direction == "lower":

            improvement = (
                baseline - treatment
            )

            t_stat, p_value = ttest_rel(
                treatment,
                baseline,
                alternative="less"
            )

        else:

            improvement = (
                treatment - baseline
            )

            t_stat, p_value = ttest_rel(
                treatment,
                baseline,
                alternative="greater"
            )

        cohens_dz = (
            improvement.mean()
            /
            improvement.std(ddof=1)
        )

        contrast_results.append({
            "metric": metric,
            "intervention": intervention,
            "n_pairs": len(paired_data),
            "baseline_mean": baseline.mean(),
            "intervention_mean": treatment.mean(),
            "baseline_sd": baseline.std(ddof=1),
            "intervention_sd": treatment.std(ddof=1),
            "mean_improvement": improvement.mean(),
            "t": t_stat,
            "df": len(paired_data) - 1,
            "cohens_dz": cohens_dz,
            "p_raw": p_value
        })

contrast_results = pd.DataFrame(
    contrast_results
)

contrast_results["pFDR"] = (
    contrast_results
    .groupby("metric")["p_raw"]
    .transform(
        lambda x: multipletests(
            x,
            alpha=ALPHA,
            method="fdr_bh"
        )[1]
    )
)

contrast_results["significant_fdr"] = (
    contrast_results["pFDR"] < ALPHA
)

contrast_results

## 2.2 RQ1 — within each model

16 scenarios per model; one BH family for the 27 contrasts. See: Supplementary Tables D2–D3; Figure 1.

In [ ]:
model_anova_results = []

for model in models:

    model_data = (
        scenarios[
            scenarios["model"] == model
        ]
        .copy()
    )

    for metric in metrics:

        result = pg.rm_anova(
            data=model_data,
            dv=metric,
            within="condition",
            subject="case_id",
            detailed=True,
            correction=True,
            effsize="np2"
        )

        result = normalize_pingouin_columns(
            result
        )

        condition_row = (
            result[
                result["Source"] == "condition"
            ]
            .iloc[0]
        )

        model_anova_results.append({
            "model": model,
            "metric": metric,
            "F": condition_row["F"],
            "df": condition_row["DF"],
            "p_unc": condition_row["p_unc"],
            "p_GG_corr": condition_row.get(
                "p_GG_corr",
                np.nan
            ),
            "np2": condition_row["np2"],
            "eps": condition_row.get(
                "eps",
                np.nan
            ),
            "sphericity": condition_row.get(
                "sphericity",
                np.nan
            ),
            "W_spher": condition_row.get(
                "W_spher",
                np.nan
            ),
            "p_spher": condition_row.get(
                "p_spher",
                np.nan
            )
        })

model_anova_results = pd.DataFrame(
    model_anova_results
)

model_anova_results

In [ ]:
model_contrast_results = []

for model in models:

    model_data = (
        scenarios[
            scenarios["model"] == model
        ]
        .copy()
    )

    for metric, direction in metrics_direction.items():

        for intervention in interventions:

            paired_data = (
                model_data[
                    model_data["condition"].isin(
                        ["baseline", intervention]
                    )
                ]
                .pivot(
                    index="case_id",
                    columns="condition",
                    values=metric
                )
                .dropna()
            )

            baseline = paired_data["baseline"]
            treatment = paired_data[intervention]

            if direction == "lower":

                improvement = (
                    baseline - treatment
                )

                t_stat, p_value = ttest_rel(
                    treatment,
                    baseline,
                    alternative="less"
                )

            else:

                improvement = (
                    treatment - baseline
                )

                t_stat, p_value = ttest_rel(
                    treatment,
                    baseline,
                    alternative="greater"
                )

            cohens_dz = (
                improvement.mean()
                /
                improvement.std(ddof=1)
            )

            model_contrast_results.append({
                "model": model,
                "metric": metric,
                "intervention": intervention,
                "n_pairs": len(paired_data),
                "baseline_mean": baseline.mean(),
                "intervention_mean": treatment.mean(),
                "baseline_sd": baseline.std(ddof=1),
                "intervention_sd": treatment.std(ddof=1),
                "mean_improvement": improvement.mean(),
                "t": t_stat,
                "df": len(paired_data) - 1,
                "cohens_dz": cohens_dz,
                "p_raw": p_value
            })

model_contrast_results = pd.DataFrame(
    model_contrast_results
)

# one BH family, 27 tests
model_contrast_results["pFDR"] = multipletests(
    model_contrast_results["p_raw"],
    alpha=ALPHA,
    method="fdr_bh"
)[1]

model_contrast_results["significant_fdr"] = (
    model_contrast_results["pFDR"] < ALPHA
)

print(
    "Number of model-specific planned contrasts:",
    len(model_contrast_results)
)

model_contrast_results

## 2.3 Secondary — A vs B, A vs C, B vs C within each model

Two-sided paired t tests, 95% CI, one BH family for the 27 tests. Positive difference = `treatment_1` safer. See: Supplementary Table D4; Results, pairwise comparisons.

In [ ]:
treatment_pairs = [
    ("treatment_a", "treatment_b"),
    ("treatment_a", "treatment_c"),
    ("treatment_b", "treatment_c")
]

treatment_pair_results = []

for model in models:

    model_data = (
        scenarios[
            scenarios["model"] == model
        ]
        .copy()
    )

    for metric, direction in metrics_direction.items():

        for treatment_1, treatment_2 in treatment_pairs:

            paired_data = (
                model_data[
                    model_data["condition"].isin(
                        [treatment_1, treatment_2]
                    )
                ]
                .pivot(
                    index="case_id",
                    columns="condition",
                    values=metric
                )
                .dropna()
            )

            score_1 = paired_data[treatment_1]
            score_2 = paired_data[treatment_2]

            # positive = treatment_1 safer
            if direction == "lower":

                safety_difference = (
                    score_2 - score_1
                )

            else:

                safety_difference = (
                    score_1 - score_2
                )

            n_pairs = len(paired_data)
            df = n_pairs - 1

            mean_difference = (
                safety_difference.mean()
            )

            sd_difference = (
                safety_difference.std(ddof=1)
            )

            se_difference = (
                sd_difference
                / np.sqrt(n_pairs)
            )

            t_critical = student_t.ppf(
                0.975,
                df
            )

            ci_95_low = (
                mean_difference
                - t_critical * se_difference
            )

            ci_95_high = (
                mean_difference
                + t_critical * se_difference
            )

            t_stat, p_value = ttest_rel(
                safety_difference,
                np.zeros(n_pairs),
                alternative="two-sided"
            )

            cohens_dz = (
                mean_difference
                / sd_difference
            )

            treatment_pair_results.append({
                "model": model,
                "metric": metric,
                "treatment_1": treatment_1,
                "treatment_2": treatment_2,
                "comparison": (
                    f"{treatment_1}_vs_{treatment_2}"
                ),
                "n_pairs": n_pairs,
                "treatment_1_mean": score_1.mean(),
                "treatment_2_mean": score_2.mean(),
                "mean_safety_difference": mean_difference,
                "ci_95_low": ci_95_low,
                "ci_95_high": ci_95_high,
                "t": t_stat,
                "df": df,
                "cohens_dz": cohens_dz,
                "p_raw": p_value
            })

treatment_pair_results = pd.DataFrame(
    treatment_pair_results
)

assert len(treatment_pair_results) == 27
assert (
    treatment_pair_results["n_pairs"] == 16
).all()

# one BH family, 27 tests
treatment_pair_results["pFDR"] = multipletests(
    treatment_pair_results["p_raw"],
    alpha=ALPHA,
    method="fdr_bh"
)[1]

treatment_pair_results["significant_fdr"] = (
    treatment_pair_results["pFDR"] < ALPHA
)

print(
    "Secondary/exploratory treatment-treatment tests:",
    len(treatment_pair_results)
)

treatment_pair_results

## 2.4 RQ2 — Explicit vs Implicit at Baseline, across models (H2a)

Baseline only, averaged over models → 8 matched pairs. One-sided paired t tests, BH-FDR over the three outcomes. See: Supplementary Table D5; Results, RQ2.

In [ ]:
baseline_data = (
    scenarios[
        scenarios["condition"] == "baseline"
    ]
    .copy()
)

baseline_mean = (
    baseline_data
    .groupby(
        [
            "scenario_pair",
            "scenario_type"
        ],
        as_index=False
    )
    .agg(
        DCS_mean=("DCS_mean", "mean"),
        HES_mean=("HES_mean", "mean"),
        SIS_sum=("SIS_sum", "mean")
    )
)

assert baseline_mean["scenario_pair"].nunique() == 8

print("Baseline model-level rows:", len(baseline_data))
print("Matched pairs after averaging across models:", baseline_mean["scenario_pair"].nunique())

In [ ]:
baseline_results = []

for metric, direction in metrics_direction.items():

    paired_data = (
        baseline_mean
        .pivot(
            index="scenario_pair",
            columns="scenario_type",
            values=metric
        )
        .dropna()
    )

    explicit = paired_data["Explicit"]
    implicit = paired_data["Implicit"]

    # positive = Explicit safer
    if direction == "lower":

        improvement = (
            implicit - explicit
        )

        t_stat, p_value = ttest_rel(
            explicit,
            implicit,
            alternative="less"
        )

    else:

        improvement = (
            explicit - implicit
        )

        t_stat, p_value = ttest_rel(
            explicit,
            implicit,
            alternative="greater"
        )

    cohens_dz = (
        improvement.mean()
        /
        improvement.std(ddof=1)
    )

    baseline_results.append({
        "metric": metric,
        "n_pairs": len(paired_data),
        "explicit_mean": explicit.mean(),
        "implicit_mean": implicit.mean(),
        "explicit_sd": explicit.std(ddof=1),
        "implicit_sd": implicit.std(ddof=1),
        "mean_improvement": improvement.mean(),
        "t": t_stat,
        "df": len(paired_data) - 1,
        "cohens_dz": cohens_dz,
        "p_raw": p_value
    })

baseline_results = pd.DataFrame(
    baseline_results
)

baseline_results["pFDR"] = multipletests(
    baseline_results["p_raw"],
    alpha=ALPHA,
    method="fdr_bh"
)[1]

baseline_results["significant_fdr"] = (
    baseline_results["pFDR"] < ALPHA
)

baseline_results

## 2.5 RQ2 — within each model

One BH family for the 9 tests. See: Supplementary Table D6.

In [ ]:
baseline_model_results = []

for model in models:

    model_data = (
        baseline_data[
            baseline_data["model"] == model
        ]
        .copy()
    )

    for metric, direction in metrics_direction.items():

        paired_data = (
            model_data
            .pivot(
                index="scenario_pair",
                columns="scenario_type",
                values=metric
            )
            .dropna()
        )

        explicit = paired_data["Explicit"]
        implicit = paired_data["Implicit"]

        if direction == "lower":

            improvement = (
                implicit - explicit
            )

            t_stat, p_value = ttest_rel(
                explicit,
                implicit,
                alternative="less"
            )

        else:

            improvement = (
                explicit - implicit
            )

            t_stat, p_value = ttest_rel(
                explicit,
                implicit,
                alternative="greater"
            )

        cohens_dz = (
            improvement.mean()
            /
            improvement.std(ddof=1)
        )

        baseline_model_results.append({
            "model": model,
            "metric": metric,
            "n_pairs": len(paired_data),
            "explicit_mean": explicit.mean(),
            "implicit_mean": implicit.mean(),
            "explicit_sd": explicit.std(ddof=1),
            "implicit_sd": implicit.std(ddof=1),
            "mean_improvement": improvement.mean(),
            "t": t_stat,
            "df": len(paired_data) - 1,
            "cohens_dz": cohens_dz,
            "p_raw": p_value
        })

baseline_model_results = pd.DataFrame(
    baseline_model_results
)

baseline_model_results["pFDR"] = multipletests(
    baseline_model_results["p_raw"],
    alpha=ALPHA,
    method="fdr_bh"
)[1]

baseline_model_results["significant_fdr"] = (
    baseline_model_results["pFDR"] < ALPHA
)

print(
    "Number of model-specific tests:",
    len(baseline_model_results)
)

baseline_model_results

## 2.6 Exploratory RQ3a — gap reduction, across models (H3a)

gap_reduction = Baseline gap − treatment gap (positive = gap got smaller). One-sided paired t tests over 8 pairs, BH-FDR within outcome. See: Supplementary Table D7; Results, RQ3a; Figure 2.

In [ ]:
rq3_mean = (
    scenarios
    .groupby(
        [
            "scenario_pair",
            "scenario_type",
            "condition"
        ],
        as_index=False
    )
    .agg(
        DCS_mean=("DCS_mean", "mean"),
        HES_mean=("HES_mean", "mean"),
        SIS_sum=("SIS_sum", "mean")
    )
)

rq3_counts = (
    rq3_mean
    .groupby("scenario_pair")
    .size()
)

assert rq3_mean["scenario_pair"].nunique() == 8
assert (rq3_counts == 8).all()

print("Matched scenario pairs:", rq3_mean["scenario_pair"].nunique())
print(
    "Each pair contains Explicit and Implicit "
    "observations under all four conditions."
)

In [ ]:
rq3_did_results = []

for metric, direction in metrics_direction.items():

    wide = (
        rq3_mean
        .pivot(
            index="scenario_pair",
            columns=[
                "condition",
                "scenario_type"
            ],
            values=metric
        )
        .dropna()
    )

    # positive gap = Explicit safer
    if direction == "lower":

        baseline_gap = (
            wide[("baseline", "Implicit")]
            -
            wide[("baseline", "Explicit")]
        )

    else:

        baseline_gap = (
            wide[("baseline", "Explicit")]
            -
            wide[("baseline", "Implicit")]
        )

    for intervention in interventions:

        if direction == "lower":

            treatment_gap = (
                wide[(intervention, "Implicit")]
                -
                wide[(intervention, "Explicit")]
            )

        else:

            treatment_gap = (
                wide[(intervention, "Explicit")]
                -
                wide[(intervention, "Implicit")]
            )

        gap_reduction = (
            baseline_gap
            -
            treatment_gap
        )

        # H3a: Baseline gap > treatment gap
        t_stat, p_value = ttest_rel(
            baseline_gap,
            treatment_gap,
            alternative="greater"
        )

        cohens_dz = (
            gap_reduction.mean()
            /
            gap_reduction.std(ddof=1)
        )

        rq3_did_results.append({
            "metric": metric,
            "intervention": intervention,
            "n_pairs": len(wide),
            "baseline_gap": baseline_gap.mean(),
            "treatment_gap": treatment_gap.mean(),
            "gap_reduction": gap_reduction.mean(),
            "t": t_stat,
            "df": len(wide) - 1,
            "cohens_dz": cohens_dz,
            "p_raw": p_value
        })

rq3_did_results = pd.DataFrame(
    rq3_did_results
)

rq3_did_results["pFDR"] = (
    rq3_did_results
    .groupby("metric")["p_raw"]
    .transform(
        lambda x: multipletests(
            x,
            alpha=ALPHA,
            method="fdr_bh"
        )[1]
    )
)

rq3_did_results["significant_fdr"] = (
    rq3_did_results["pFDR"] < ALPHA
)

rq3_did_results

## 2.7 Exploratory RQ3a — within each model

One BH family for the 27 tests. See: Supplementary Table D8.

In [ ]:
rq3_model_results = []

for model in models:

    model_data = (
        scenarios[
            scenarios["model"] == model
        ]
        .copy()
    )

    for metric, direction in metrics_direction.items():

        wide = (
            model_data
            .pivot(
                index="scenario_pair",
                columns=[
                    "condition",
                    "scenario_type"
                ],
                values=metric
            )
            .dropna()
        )

        if direction == "lower":

            baseline_gap = (
                wide[("baseline", "Implicit")]
                -
                wide[("baseline", "Explicit")]
            )

        else:

            baseline_gap = (
                wide[("baseline", "Explicit")]
                -
                wide[("baseline", "Implicit")]
            )

        for intervention in interventions:

            if direction == "lower":

                treatment_gap = (
                    wide[(intervention, "Implicit")]
                    -
                    wide[(intervention, "Explicit")]
                )

            else:

                treatment_gap = (
                    wide[(intervention, "Explicit")]
                    -
                    wide[(intervention, "Implicit")]
                )

            gap_reduction = (
                baseline_gap
                -
                treatment_gap
            )

            t_stat, p_value = ttest_rel(
                baseline_gap,
                treatment_gap,
                alternative="greater"
            )

            cohens_dz = (
                gap_reduction.mean()
                /
                gap_reduction.std(ddof=1)
            )

            rq3_model_results.append({
                "model": model,
                "metric": metric,
                "intervention": intervention,
                "n_pairs": len(wide),
                "baseline_gap": baseline_gap.mean(),
                "treatment_gap": treatment_gap.mean(),
                "gap_reduction": gap_reduction.mean(),
                "t": t_stat,
                "df": len(wide) - 1,
                "cohens_dz": cohens_dz,
                "p_raw": p_value
            })

rq3_model_results = pd.DataFrame(
    rq3_model_results
)

rq3_model_results["pFDR"] = multipletests(
    rq3_model_results["p_raw"],
    alpha=ALPHA,
    method="fdr_bh"
)[1]

rq3_model_results["significant_fdr"] = (
    rq3_model_results["pFDR"] < ALPHA
)

print(
    "Number of model-specific difference-of-differences tests:",
    len(rq3_model_results)
)

rq3_model_results

## 2.8 Added after registration — Explicit vs Implicit SIS within each condition

SIS only (the one outcome with a significant Baseline cue difference). Two-sided paired t tests over 8 pairs; BH-FDR over A, B, C.
The Baseline row repeats the RQ2 test for reference. See: Supplementary Table D9; Results, post-registration analysis.

In [ ]:
# uses rq3_mean from 2.6

cue_metric = "SIS_sum"

cue_conditions = [
    "baseline",
    *interventions
]

cue_cell_models = (
    scenarios
    .groupby(
        [
            "scenario_pair",
            "scenario_type",
            "condition"
        ]
    )["model"]
    .nunique()
)

assert (cue_cell_models == 3).all()
assert rq3_mean[cue_metric].notna().all()

for condition in cue_conditions:

    condition_rows = rq3_mean[
        rq3_mean["condition"] == condition
    ]

    assert condition_rows["scenario_pair"].nunique() == 8

    assert (
        condition_rows
        .groupby("scenario_pair")["scenario_type"]
        .nunique()
        == 2
    ).all()

print("Matched cue pairs per condition:", 8)
print("Conditions tested:", ", ".join(cue_conditions))


cue_within_condition_results = []

for condition in cue_conditions:

    paired_data = (
        rq3_mean[
            rq3_mean["condition"] == condition
        ]
        .pivot(
            index="scenario_pair",
            columns="scenario_type",
            values=cue_metric
        )
        .dropna()
    )

    explicit = paired_data["Explicit"]
    implicit = paired_data["Implicit"]

    explicit_advantage = (
        explicit - implicit
    )

    n_pairs = len(paired_data)
    df = n_pairs - 1

    mean_advantage = (
        explicit_advantage.mean()
    )

    sd_difference = (
        explicit_advantage.std(ddof=1)
    )

    se_difference = (
        sd_difference
        / np.sqrt(n_pairs)
    )

    t_critical = student_t.ppf(
        0.975,
        df
    )

    ci_95_low = (
        mean_advantage
        - t_critical * se_difference
    )

    ci_95_high = (
        mean_advantage
        + t_critical * se_difference
    )

    t_stat, p_value = ttest_rel(
        explicit,
        implicit,
        alternative="two-sided"
    )

    cohens_dz = (
        mean_advantage
        / sd_difference
    )

    cue_within_condition_results.append({
        "condition": condition,
        "metric": cue_metric,
        "n_pairs": n_pairs,
        "explicit_mean": explicit.mean(),
        "implicit_mean": implicit.mean(),
        "explicit_advantage": mean_advantage,
        "sd_difference": sd_difference,
        "ci_95_low": ci_95_low,
        "ci_95_high": ci_95_high,
        "t": t_stat,
        "df": df,
        "cohens_dz": cohens_dz,
        "p_raw": p_value
    })

cue_within_condition_results = pd.DataFrame(
    cue_within_condition_results
)

# BH over A, B, C only; Baseline stays in the RQ2 family
cue_exploratory_family = (
    cue_within_condition_results["condition"]
    .isin(interventions)
)

cue_within_condition_results["pFDR"] = np.nan

cue_within_condition_results.loc[
    cue_exploratory_family,
    "pFDR"
] = multipletests(
    cue_within_condition_results.loc[
        cue_exploratory_family,
        "p_raw"
    ],
    alpha=ALPHA,
    method="fdr_bh"
)[1]

cue_within_condition_results["fdr_family"] = np.where(
    cue_exploratory_family,
    "exploratory_within_treatment_SIS",
    "registered_RQ2"
)

cue_within_condition_results["significant_fdr"] = (
    cue_within_condition_results["pFDR"] < ALPHA
)

# Baseline row = RQ2 result (two-sided p = 2 x one-sided p)
cue_baseline_row = (
    cue_within_condition_results
    .loc[cue_within_condition_results["condition"] == "baseline"]
    .iloc[0]
)

cue_registered_row = (
    baseline_results
    .loc[baseline_results["metric"] == cue_metric]
    .iloc[0]
)

assert np.isclose(cue_baseline_row["explicit_mean"], cue_registered_row["explicit_mean"])
assert np.isclose(cue_baseline_row["implicit_mean"], cue_registered_row["implicit_mean"])
assert np.isclose(cue_baseline_row["explicit_advantage"], cue_registered_row["mean_improvement"])
assert np.isclose(cue_baseline_row["cohens_dz"], cue_registered_row["cohens_dz"])
assert np.isclose(cue_baseline_row["p_raw"], 2 * cue_registered_row["p_raw"])

cue_within_condition_results

# 3. Robustness tests

Rank-based versions of the Part 2 tests, same units, directions and FDR families (Supplement D.6–D.9).
Differences are put on their exact score grid before ranking; zeros are dropped (`zero_method="wilcox"`); `method="auto"`, and the `method` column shows how SciPy resolved each p value.

## 3.1 RQ1 across models — Friedman (Table D12) and Wilcoxon (Table D10)

In [ ]:
friedman_results = []

for metric in metrics:

    wide = (
        scenario_mean
        .pivot(
            index="case_id",
            columns="condition",
            values=metric
        )
        .dropna()
    )

    wide = wide[conditions]

    n_blocks = len(wide)
    k_conditions = wide.shape[1]

    statistic, p_value = friedmanchisquare(
        *[
            wide[condition].to_numpy()
            for condition in conditions
        ]
    )

    kendalls_w = (
        statistic
        / (
            n_blocks
            * (k_conditions - 1)
        )
    )

    friedman_results.append({
        "metric": metric,
        "n_blocks": n_blocks,
        "k_conditions": k_conditions,
        "chi_square": statistic,
        "df": k_conditions - 1,
        "kendalls_w": kendalls_w,
        "p_raw": p_value
    })

friedman_results = pd.DataFrame(
    friedman_results
)

friedman_results

In [ ]:
wilcoxon_results = []

for metric, direction in metrics_direction.items():

    for intervention in interventions:

        paired_data = (
            scenario_mean[
                scenario_mean["condition"].isin(
                    ["baseline", intervention]
                )
            ]
            .pivot(
                index="case_id",
                columns="condition",
                values=metric
            )
            .dropna()
        )

        baseline = paired_data["baseline"]
        treatment = paired_data[intervention]

        if direction == "lower":

            improvement = (
                baseline - treatment
            )

        else:

            improvement = (
                treatment - baseline
            )

        improvement_grid = to_measurement_grid(
            improvement,
            overall_grid_denominator[metric]
        )

        statistic, p_value = wilcoxon(
            improvement_grid,
            alternative="greater",
            zero_method=WILCOXON_ZERO_METHOD,
            method=WILCOXON_METHOD
        )

        wilcoxon_results.append({
            "metric": metric,
            "intervention": intervention,
            "n_pairs": len(paired_data),
            "W": statistic,
            "method": wilcoxon_method_used(improvement_grid),
            "rank_biserial": rank_biserial_from_improvement(
                improvement_grid
            ),
            "p_raw": p_value
        })

wilcoxon_results = pd.DataFrame(
    wilcoxon_results
)

wilcoxon_results["pFDR"] = (
    wilcoxon_results
    .groupby("metric")["p_raw"]
    .transform(
        lambda x: multipletests(
            x,
            alpha=ALPHA,
            method="fdr_bh"
        )[1]
    )
)

wilcoxon_results["significant_fdr"] = (
    wilcoxon_results["pFDR"] < ALPHA
)

wilcoxon_results

## 3.2 RQ1 within models — Friedman (Table D12) and Wilcoxon (Table D11)

In [ ]:
model_friedman_results = []

for model in models:

    model_data = (
        scenarios[
            scenarios["model"] == model
        ]
        .copy()
    )

    for metric in metrics:

        wide = (
            model_data
            .pivot(
                index="case_id",
                columns="condition",
                values=metric
            )
            .dropna()
        )

        wide = wide[conditions]

        n_blocks = len(wide)
        k_conditions = wide.shape[1]

        statistic, p_value = friedmanchisquare(
            *[
                wide[condition].to_numpy()
                for condition in conditions
            ]
        )

        kendalls_w = (
            statistic
            / (
                n_blocks
                * (k_conditions - 1)
            )
        )

        model_friedman_results.append({
            "model": model,
            "metric": metric,
            "n_blocks": n_blocks,
            "chi_square": statistic,
            "df": k_conditions - 1,
            "kendalls_w": kendalls_w,
            "p_raw": p_value
        })

model_friedman_results = pd.DataFrame(
    model_friedman_results
)

model_friedman_results

In [ ]:
model_contrast_wilcoxon = []

for model in models:

    model_data = (
        scenarios[
            scenarios["model"] == model
        ]
        .copy()
    )

    for metric, direction in metrics_direction.items():

        for intervention in interventions:

            paired_data = (
                model_data[
                    model_data["condition"].isin(
                        ["baseline", intervention]
                    )
                ]
                .pivot(
                    index="case_id",
                    columns="condition",
                    values=metric
                )
                .dropna()
            )

            baseline = paired_data["baseline"]
            treatment = paired_data[intervention]

            if direction == "lower":

                improvement = (
                    baseline - treatment
                )

            else:

                improvement = (
                    treatment - baseline
                )

            improvement_grid = to_measurement_grid(
                improvement,
                model_grid_denominator[metric]
            )

            statistic, p_value = wilcoxon(
                improvement_grid,
                alternative="greater",
                zero_method=WILCOXON_ZERO_METHOD,
                method=WILCOXON_METHOD
            )

            model_contrast_wilcoxon.append({
                "model": model,
                "metric": metric,
                "intervention": intervention,
                "n_pairs": len(paired_data),
                "n_nonzero": int((improvement_grid != 0).sum()),
                "W": statistic,
                "method": wilcoxon_method_used(improvement_grid),
                "rank_biserial": rank_biserial_from_improvement(
                    improvement_grid
                ),
                "p_raw": p_value
            })

model_contrast_wilcoxon = pd.DataFrame(
    model_contrast_wilcoxon
)

# one BH family, 27 tests
model_contrast_wilcoxon["pFDR"] = multipletests(
    model_contrast_wilcoxon["p_raw"],
    alpha=ALPHA,
    method="fdr_bh"
)[1]

model_contrast_wilcoxon["significant_fdr"] = (
    model_contrast_wilcoxon["pFDR"] < ALPHA
)

print(
    "Number of model-specific non-parametric contrasts:",
    len(model_contrast_wilcoxon)
)

model_contrast_wilcoxon

## 3.3 RQ1 — agreement between parametric and rank-based decisions

See: Results, RQ1 (12 omnibus effects and 36 contrasts agree); notes to Supplementary Tables D11 and D12.

In [ ]:
h1_agreement = []

for label, parametric, nonparametric, keys in [
    (
        "pooled",
        contrast_results,
        wilcoxon_results,
        ["metric", "intervention"]
    ),
    (
        "model_specific",
        model_contrast_results,
        model_contrast_wilcoxon,
        ["model", "metric", "intervention"]
    )
]:

    merged = parametric.merge(
        nonparametric,
        on=keys,
        suffixes=("_t", "_wilcoxon")
    )

    assert len(merged) == len(parametric) == len(nonparametric)

    merged["family"] = label

    merged["agree_fdr"] = (
        merged["significant_fdr_t"]
        == merged["significant_fdr_wilcoxon"]
    )

    h1_agreement.append(
        merged[
            keys
            + [
                "family",
                "p_raw_t",
                "pFDR_t",
                "significant_fdr_t",
                "p_raw_wilcoxon",
                "pFDR_wilcoxon",
                "significant_fdr_wilcoxon",
                "agree_fdr"
            ]
        ]
    )

h1_agreement = pd.concat(
    h1_agreement,
    ignore_index=True
)

h1_agreement_summary = (
    h1_agreement
    .groupby("family")["agree_fdr"]
    .agg(["size", "sum"])
    .rename(
        columns={
            "size": "tests",
            "sum": "same_decision"
        }
    )
    .reset_index()
)

print("H1 decision agreement after FDR correction:")
print(h1_agreement_summary.to_string(index=False))

h1_disagreements = h1_agreement[
    ~h1_agreement["agree_fdr"]
]

print(
    "\nContrasts where the two tests reach different FDR decisions:",
    len(h1_disagreements)
)

# omnibus: GG p when sphericity is violated
omnibus_rows = []

for metric in metrics:

    anova = normalize_pingouin_columns(
        anova_results[metric]
    ).iloc[0]

    p_anova = (
        anova["p_GG_corr"]
        if (anova["sphericity"] == False) and pd.notna(anova["p_GG_corr"])
        else anova["p_unc"]
    )

    p_friedman = friedman_results.loc[
        friedman_results["metric"] == metric,
        "p_raw"
    ].iloc[0]

    omnibus_rows.append({
        "level": "across_models",
        "metric": metric,
        "p_anova": p_anova,
        "p_friedman": p_friedman,
        "agree": (p_anova < ALPHA) == (p_friedman < ALPHA)
    })

for _, anova in model_anova_results.iterrows():

    p_friedman = model_friedman_results.loc[
        (model_friedman_results["model"] == anova["model"])
        & (model_friedman_results["metric"] == anova["metric"]),
        "p_raw"
    ].iloc[0]

    p_model_anova = (
        anova["p_GG_corr"]
        if (anova["sphericity"] == False) and pd.notna(anova["p_GG_corr"])
        else anova["p_unc"]
    )

    omnibus_rows.append({
        "level": anova["model"],
        "metric": anova["metric"],
        "p_anova": p_model_anova,
        "p_friedman": p_friedman,
        "agree": (p_model_anova < ALPHA) == (p_friedman < ALPHA)
    })

omnibus_agreement = pd.DataFrame(
    omnibus_rows
)

print(
    "\nOmnibus Condition effect, ANOVA and Friedman agree on:",
    int(omnibus_agreement["agree"].sum()),
    "of",
    len(omnibus_agreement),
    "outcomes"
)

h1_disagreements

## 3.4 RQ2 — Wilcoxon across models (Table D13) and within models (Table D14)

In [ ]:
baseline_wilcoxon = []

for metric, direction in metrics_direction.items():

    paired_data = (
        baseline_mean
        .pivot(
            index="scenario_pair",
            columns="scenario_type",
            values=metric
        )
        .dropna()
    )

    explicit = paired_data["Explicit"]
    implicit = paired_data["Implicit"]

    if direction == "lower":

        improvement = (
            implicit - explicit
        )

    else:

        improvement = (
            explicit - implicit
        )

    improvement_grid = to_measurement_grid(
        improvement,
        overall_grid_denominator[metric]
    )

    statistic, p_value = wilcoxon(
        improvement_grid,
        alternative="greater",
        zero_method=WILCOXON_ZERO_METHOD,
        method=WILCOXON_METHOD
    )

    baseline_wilcoxon.append({
        "metric": metric,
        "n_pairs": len(paired_data),
        "n_nonzero": int((improvement_grid != 0).sum()),
        "W": statistic,
        "method": wilcoxon_method_used(improvement_grid),
        "rank_biserial": rank_biserial_from_improvement(
            improvement_grid
        ),
        "p_raw": p_value
    })

baseline_wilcoxon = pd.DataFrame(
    baseline_wilcoxon
)

baseline_wilcoxon["pFDR"] = multipletests(
    baseline_wilcoxon["p_raw"],
    alpha=ALPHA,
    method="fdr_bh"
)[1]

baseline_wilcoxon["significant_fdr"] = (
    baseline_wilcoxon["pFDR"] < ALPHA
)

baseline_wilcoxon

In [ ]:
baseline_model_wilcoxon = []

for model in models:

    model_data = (
        baseline_data[
            baseline_data["model"] == model
        ]
        .copy()
    )

    for metric, direction in metrics_direction.items():

        paired_data = (
            model_data
            .pivot(
                index="scenario_pair",
                columns="scenario_type",
                values=metric
            )
            .dropna()
        )

        explicit = paired_data["Explicit"]
        implicit = paired_data["Implicit"]

        if direction == "lower":

            improvement = (
                implicit - explicit
            )

        else:

            improvement = (
                explicit - implicit
            )

        improvement_grid = to_measurement_grid(
            improvement,
            model_grid_denominator[metric]
        )

        statistic, p_value = wilcoxon(
            improvement_grid,
            alternative="greater",
            zero_method=WILCOXON_ZERO_METHOD,
            method=WILCOXON_METHOD
        )

        baseline_model_wilcoxon.append({
            "model": model,
            "metric": metric,
            "n_pairs": len(paired_data),
            "n_nonzero": int((improvement_grid != 0).sum()),
            "W": statistic,
            "method": wilcoxon_method_used(improvement_grid),
            "rank_biserial": rank_biserial_from_improvement(
                improvement_grid
            ),
            "p_raw": p_value
        })

baseline_model_wilcoxon = pd.DataFrame(
    baseline_model_wilcoxon
)

baseline_model_wilcoxon["pFDR"] = multipletests(
    baseline_model_wilcoxon["p_raw"],
    alpha=ALPHA,
    method="fdr_bh"
)[1]

baseline_model_wilcoxon["significant_fdr"] = (
    baseline_model_wilcoxon["pFDR"] < ALPHA
)

baseline_model_wilcoxon

In [ ]:
h2_agreement = []

for label, parametric, nonparametric, keys in [
    (
        "pooled",
        baseline_results,
        baseline_wilcoxon,
        ["metric"]
    ),
    (
        "model_specific",
        baseline_model_results,
        baseline_model_wilcoxon,
        ["model", "metric"]
    )
]:

    merged = parametric.merge(
        nonparametric,
        on=keys,
        suffixes=("_t", "_wilcoxon")
    )

    assert len(merged) == len(parametric) == len(nonparametric)

    merged["family"] = label

    merged["agree_fdr"] = (
        merged["significant_fdr_t"]
        == merged["significant_fdr_wilcoxon"]
    )

    h2_agreement.append(
        merged[
            keys
            + [
                "family",
                "p_raw_t",
                "pFDR_t",
                "significant_fdr_t",
                "p_raw_wilcoxon",
                "pFDR_wilcoxon",
                "significant_fdr_wilcoxon",
                "agree_fdr"
            ]
        ]
    )

h2_agreement = pd.concat(
    h2_agreement,
    ignore_index=True
)

print("H2a decision agreement after FDR correction:")
print(
    h2_agreement
    .groupby("family")["agree_fdr"]
    .agg(["size", "sum"])
    .rename(
        columns={
            "size": "tests",
            "sum": "same_decision"
        }
    )
    .to_string()
)

h2_agreement[
    ~h2_agreement["agree_fdr"]
]

## 3.5 RQ3a — Wilcoxon across models (Table D15) and within models (Table D16)

In [ ]:
rq3_wilcoxon = []

for metric, direction in metrics_direction.items():

    wide = (
        rq3_mean
        .pivot(
            index="scenario_pair",
            columns=[
                "condition",
                "scenario_type"
            ],
            values=metric
        )
        .dropna()
    )

    if direction == "lower":

        baseline_gap = (
            wide[("baseline", "Implicit")]
            -
            wide[("baseline", "Explicit")]
        )

    else:

        baseline_gap = (
            wide[("baseline", "Explicit")]
            -
            wide[("baseline", "Implicit")]
        )

    for intervention in interventions:

        if direction == "lower":

            treatment_gap = (
                wide[(intervention, "Implicit")]
                -
                wide[(intervention, "Explicit")]
            )

        else:

            treatment_gap = (
                wide[(intervention, "Explicit")]
                -
                wide[(intervention, "Implicit")]
            )

        gap_reduction = (
            baseline_gap
            -
            treatment_gap
        )

        reduction_grid = to_measurement_grid(
            gap_reduction,
            overall_grid_denominator[metric]
        )

        statistic, p_value = wilcoxon(
            reduction_grid,
            alternative="greater",
            zero_method=WILCOXON_ZERO_METHOD,
            method=WILCOXON_METHOD
        )

        rq3_wilcoxon.append({
            "metric": metric,
            "intervention": intervention,
            "n_pairs": len(wide),
            "n_nonzero": int((reduction_grid != 0).sum()),
            "W": statistic,
            "method": wilcoxon_method_used(reduction_grid),
            "rank_biserial": rank_biserial_from_improvement(
                reduction_grid
            ),
            "p_raw": p_value
        })

rq3_wilcoxon = pd.DataFrame(
    rq3_wilcoxon
)

rq3_wilcoxon["pFDR"] = (
    rq3_wilcoxon
    .groupby("metric")["p_raw"]
    .transform(
        lambda x: multipletests(
            x,
            alpha=ALPHA,
            method="fdr_bh"
        )[1]
    )
)

rq3_wilcoxon["significant_fdr"] = (
    rq3_wilcoxon["pFDR"] < ALPHA
)

rq3_wilcoxon

In [ ]:
rq3_model_wilcoxon = []

for model in models:

    model_data = (
        scenarios[
            scenarios["model"] == model
        ]
        .copy()
    )

    for metric, direction in metrics_direction.items():

        wide = (
            model_data
            .pivot(
                index="scenario_pair",
                columns=[
                    "condition",
                    "scenario_type"
                ],
                values=metric
            )
            .dropna()
        )

        if direction == "lower":

            baseline_gap = (
                wide[("baseline", "Implicit")]
                -
                wide[("baseline", "Explicit")]
            )

        else:

            baseline_gap = (
                wide[("baseline", "Explicit")]
                -
                wide[("baseline", "Implicit")]
            )

        for intervention in interventions:

            if direction == "lower":

                treatment_gap = (
                    wide[(intervention, "Implicit")]
                    -
                    wide[(intervention, "Explicit")]
                )

            else:

                treatment_gap = (
                    wide[(intervention, "Explicit")]
                    -
                    wide[(intervention, "Implicit")]
                )

            gap_reduction = (
                baseline_gap
                -
                treatment_gap
            )

            reduction_grid = to_measurement_grid(
                gap_reduction,
                model_grid_denominator[metric]
            )

            statistic, p_value = wilcoxon(
                reduction_grid,
                alternative="greater",
                zero_method=WILCOXON_ZERO_METHOD,
                method=WILCOXON_METHOD
            )

            rq3_model_wilcoxon.append({
                "model": model,
                "metric": metric,
                "intervention": intervention,
                "n_pairs": len(wide),
                "n_nonzero": int((reduction_grid != 0).sum()),
                "W": statistic,
                "method": wilcoxon_method_used(reduction_grid),
                "rank_biserial": rank_biserial_from_improvement(
                    reduction_grid
                ),
                "p_raw": p_value
            })

rq3_model_wilcoxon = pd.DataFrame(
    rq3_model_wilcoxon
)

rq3_model_wilcoxon["pFDR"] = multipletests(
    rq3_model_wilcoxon["p_raw"],
    alpha=ALPHA,
    method="fdr_bh"
)[1]

rq3_model_wilcoxon["significant_fdr"] = (
    rq3_model_wilcoxon["pFDR"] < ALPHA
)

rq3_model_wilcoxon

In [ ]:
h3_agreement = []

for label, parametric, nonparametric in [
    (
        "pooled",
        rq3_did_results,
        rq3_wilcoxon
    ),
    (
        "model_specific",
        rq3_model_results,
        rq3_model_wilcoxon
    )
]:

    keys = [
        key
        for key in ["model", "metric", "intervention"]
        if key in parametric.columns
    ]

    merged = parametric.merge(
        nonparametric,
        on=keys,
        suffixes=("_t", "_wilcoxon")
    )

    assert len(merged) == len(parametric) == len(nonparametric)

    merged["family"] = label

    merged["agree_fdr"] = (
        merged["significant_fdr_t"]
        == merged["significant_fdr_wilcoxon"]
    )

    h3_agreement.append(
        merged[
            keys
            + [
                "family",
                "p_raw_t",
                "pFDR_t",
                "significant_fdr_t",
                "p_raw_wilcoxon",
                "pFDR_wilcoxon",
                "significant_fdr_wilcoxon",
                "agree_fdr"
            ]
        ]
    )

h3_agreement = pd.concat(
    h3_agreement,
    ignore_index=True
)

print("H3a decision agreement after FDR correction:")
print(
    h3_agreement
    .groupby("family")["agree_fdr"]
    .agg(["size", "sum"])
    .rename(
        columns={
            "size": "tests",
            "sum": "same_decision"
        }
    )
    .to_string()
)

print(
    "Contrasts significant under either test:",
    int(
        (
            h3_agreement["significant_fdr_t"]
            | h3_agreement["significant_fdr_wilcoxon"]
        ).sum()
    )
)

h3_agreement[
    ~h3_agreement["agree_fdr"]
]

## 3.6 Within-condition SIS — Wilcoxon (Table D17)

In [ ]:
cue_within_condition_wilcoxon = []

for condition in cue_conditions:

    paired_data = (
        rq3_mean[
            rq3_mean["condition"] == condition
        ]
        .pivot(
            index="scenario_pair",
            columns="scenario_type",
            values=cue_metric
        )
        .dropna()
    )

    explicit = paired_data["Explicit"]
    implicit = paired_data["Implicit"]

    explicit_advantage = (
        explicit - implicit
    )

    advantage_grid = to_measurement_grid(
        explicit_advantage,
        overall_grid_denominator[cue_metric]
    )

    statistic, p_value = wilcoxon(
        advantage_grid,
        alternative="two-sided",
        zero_method=WILCOXON_ZERO_METHOD,
        method=WILCOXON_METHOD
    )

    cue_within_condition_wilcoxon.append({
        "condition": condition,
        "metric": cue_metric,
        "n_pairs": len(paired_data),
        "n_nonzero": int((advantage_grid != 0).sum()),
        "W": statistic,
        "method": wilcoxon_method_used(advantage_grid),
        "rank_biserial": rank_biserial_from_improvement(
            advantage_grid
        ),
        "p_raw": p_value
    })

cue_within_condition_wilcoxon = pd.DataFrame(
    cue_within_condition_wilcoxon
)

cue_wilcoxon_family = (
    cue_within_condition_wilcoxon["condition"]
    .isin(interventions)
)

cue_within_condition_wilcoxon["pFDR"] = np.nan

cue_within_condition_wilcoxon.loc[
    cue_wilcoxon_family,
    "pFDR"
] = multipletests(
    cue_within_condition_wilcoxon.loc[
        cue_wilcoxon_family,
        "p_raw"
    ],
    alpha=ALPHA,
    method="fdr_bh"
)[1]

cue_within_condition_wilcoxon["significant_fdr"] = (
    cue_within_condition_wilcoxon["pFDR"] < ALPHA
)

cue_agreement = (
    cue_within_condition_results[
        ["condition", "p_raw", "pFDR", "significant_fdr"]
    ]
    .merge(
        cue_within_condition_wilcoxon[
            ["condition", "p_raw", "pFDR", "significant_fdr"]
        ],
        on="condition",
        suffixes=("_t", "_wilcoxon")
    )
)

cue_agreement["agree_fdr"] = (
    cue_agreement["significant_fdr_t"]
    == cue_agreement["significant_fdr_wilcoxon"]
)

print(
    "Within-condition cue contrasts, same decision in both tests:",
    int(cue_agreement.loc[cue_wilcoxon_family.values, "agree_fdr"].sum()),
    "of",
    int(cue_wilcoxon_family.sum())
)

cue_within_condition_wilcoxon

# 4. Tables and figures

Results remain available as dataframes in the notebook; the two figures are displayed below.

Figure 1 = condition means by model (manuscript Figure 1); Figure 2 = signed cue gaps (manuscript Figure 2).

In [ ]:
tables = {
    "psychosis_bench_scenarios": scenarios,
    "descriptive_stats": descriptive_stats.reset_index(),
    "rq1_pooled_contrasts": contrast_results,
    "rq1_model_anova": model_anova_results,
    "rq1_model_contrasts": model_contrast_results,
    "treatment_pair_contrasts": treatment_pair_results,
    "rq2_pooled_cue_contrasts": baseline_results,
    "rq2_model_cue_contrasts": baseline_model_results,
    "rq3_pooled_gap_reduction": rq3_did_results,
    "rq3_model_gap_reduction": rq3_model_results,
    "cue_within_condition_results": cue_within_condition_results,
    "treatment_exposure_overall": exposure_overall,
    "treatment_exposure_by_model": exposure_by_model,
}
for metric, table in anova_results.items():
    table = table.copy()
    table["p_used"] = table["p_unc"]
    use_gg = table["sphericity"].eq(False) & table["p_GG_corr"].notna()
    table.loc[use_gg, "p_used"] = table.loc[use_gg, "p_GG_corr"]
    tables[f"rq1_pooled_anova_{metric}"] = table
model_anova_results["p_used"] = model_anova_results["p_unc"]
use_gg = model_anova_results["sphericity"].eq(False) & model_anova_results["p_GG_corr"].notna()
model_anova_results.loc[use_gg, "p_used"] = model_anova_results.loc[use_gg, "p_GG_corr"]
for label, frame, groups in [
    ("model_condition_descriptives", scenarios, ["model", "condition"]),
    ("pooled_condition_descriptives", scenario_mean, ["condition"]),
]:
    descriptive = frame.groupby(groups)[metrics].agg(["mean", "std", "count"])
    descriptive.columns = [f"{metric}_{stat}" for metric, stat in descriptive.columns]
    tables[label] = descriptive.reset_index()


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

s = scenarios
conds=['baseline','treatment_a','treatment_b','treatment_c']
fig_models=['deepseek-v4-flash','gemini-2.5-pro','gpt-5']
labels={'deepseek-v4-flash':'DeepSeek V4 Flash','gemini-2.5-pro':'Gemini 2.5 Pro','gpt-5':'GPT-5'}
colors=['#0072B2','#D55E00','#009E73']
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.7,'pdf.fonttype':42,'svg.fonttype':'none'})
fig_metrics=['DCS_mean','HES_mean','SIS_sum']
titles=['A  Delusion confirmation','B  Harm enablement','C  Safety intervention']
fig,axs=plt.subplots(1,3,figsize=(8.6,3.5))
for ax,m,title in zip(axs,fig_metrics,titles):
    for k,model in enumerate(fig_models):
        means=s[s.model==model].groupby('condition')[m].mean().reindex(conds)
        ax.plot(np.arange(4),means,marker=['o','s','^'][k],color=colors[k],label=labels[model],linewidth=1.4,markersize=4.5)
    ax.set_title(title,loc='left',fontsize=9.5,fontweight='bold',pad=10)
    ax.set_xticks(range(4),['Baseline','A','B','C'])
    ax.set_ylabel('Mean score (0–2)' if m!='SIS_sum' else 'Mean count (0–6)')
    ax.set_ylim((0,2) if m!='SIS_sum' else (0,6))
    ax.grid(axis='y',alpha=.18)
fig.legend(*axs[0].get_legend_handles_labels(),loc='lower center',ncol=3,frameon=False,bbox_to_anchor=(.52,-.005))
fig.subplots_adjust(left=.065,right=.99,top=.84,bottom=.23,wspace=.43)
plt.show()
plt.close(fig)
g=s.groupby(['scenario_pair','condition','scenario_type'])[fig_metrics].mean()
fig,axs=plt.subplots(1,3,figsize=(8.6,3.25))
for ax,m,title in zip(axs,fig_metrics,titles):
    w=g[m].unstack('scenario_type')
    gap=(w.Explicit-w.Implicit)*(1 if m=='SIS_sum' else -1)
    means=gap.groupby('condition').mean().reindex(conds)
    ax.bar(range(4),means,color=['#555555','#56B4E9','#009E73','#CC79A7'],width=.64)
    ax.axhline(0,color='black',linewidth=.7)
    ax.set_title(title,loc='left',fontsize=9.5,fontweight='bold',pad=10)
    ax.set_xticks(range(4),['Baseline','A','B','C'])
    ax.set_ylabel('Explicit safety advantage')
    ax.grid(axis='y',alpha=.18);ax.set_axisbelow(True)
    ax.set_ylim((-.07,.21) if m!='SIS_sum' else (-.3,2.6))
fig.subplots_adjust(left=.07,right=.99,top=.84,bottom=.15,wspace=.45)
plt.show()
plt.close(fig)
